# PT SJR MODEL BUILDING

### **Phase 1: Core Leach Models (Empirical + Kinetic)**
#### **1. Gold Recovery Model**
**Goal:** Predict % gold recovery over time (or per tank) based on:
* Feed grades (Au, Ag)
* Dissolved O₂ (DO)
* Cyanide concentration (CN)
* Carbon concentration
* Residence time
* Pulp density / solids
* Particle size / liberation

**Approaches:**
* Empirical (regression/ML): XGBoost or Random Forest
* Mechanistic: Kinetic model based on de Andrade Lima & Hodouin (2005)
* Hybrid: Kinetics + ML residual learning

#### **2. Cyanide Consumption Model**
**Goal:** Predict total CN usage (kg/day or kg/tonne) based on:
* Same process variables as above
* Gold and silver loading
* Losses to tailings
* Decomposition rate (pH, DO)

**Approaches:**
* Empirical regression (daily CN usage)
* Site-tuned kinetic consumption equation

#### **3. Tailings Loss Model**
**Goal:** Predict gold and CN losses to tailings (g/t or ppm), to quantify inefficiencies.

**Inputs:**
* Feed characteristics (grade, fines)
* Process conditions (DO, CN, carbon)
* Residence time, leach tank progression

**Value:** Enables economic loss tracking and recovery opportunity identification.

### **Phase 2: Scenario + Solver Tools**
#### **4. Inverse Solver**
Goal: Given a target recovery (e.g. 92%), solve for required:
* CN concentration
* DO level
* Residence time or particle size

Approach:
Use fitted kinetic model
Integrate into frontend decision-support tool

#### **5. Recommendation Engine**
**Goal:** Provide daily guidance on CN/DO targets based on:
* Historical performance
* Predicted response (recovery, consumption, loss)
* Operating constraints and trade-offs

## 📂 Load Data & Initial Setup 

In [1]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
import math
from pandas import Timestamp
import matplotlib.dates as mdates
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.utils import resample

In [ ]:
# --- Load first data file for 2024 ---
file_2024_path = "PTSJ_data.xlsx"

# Load the multi-row header from B2:AU4 (rows 1-3)
header_df = pd.read_excel(file_2024_path, header=None, skiprows=1, nrows=3, usecols="B:AV")

# Load the actual data (B5:BD39 → skiprows=4, nrows=35)
df_2024 = pd.read_excel(file_2024_path, header=None, skiprows=4, nrows=35, usecols="B:AV")

# Load the Date column separately from B5:B39
date_series = pd.read_excel(file_2024_path, header=None, skiprows=4, nrows=35, usecols="B")

# Flatten the headers
def flatten_header(row1, row2, row3):
	def clean(x): return str(x).strip().replace(" ", "_").replace("#", "mesh") if pd.notna(x) else ""
	return "_".join([clean(row1), clean(row2), clean(row3)]).strip("_")

flattened_cols = [
	flatten_header(r1, r2, r3)
	for r1, r2, r3 in zip(header_df.iloc[0], header_df.iloc[1], header_df.iloc[2])
]

# Assign headers and attach Date column
df_2024.columns = flattened_cols
if "Date" not in df_2024.columns:
	df_2024.insert(0, "Date", pd.to_datetime(date_series.iloc[:, 0]))

# --- Load second data file for 2025 ---
file_2025_path = "ptsj_2025_data.xlsx"

jan_2025 = pd.read_excel(file_2025_path, sheet_name="Jan 25", header=None, skiprows=9)
feb_2025 = pd.read_excel(file_2025_path, sheet_name="Feb 25", header=None, skiprows=9)
mar_2025 = pd.read_excel(file_2025_path, sheet_name="Mar 25", header=None, skiprows=9)
apr_2025 = pd.read_excel(file_2025_path, sheet_name="April 25", header=None, skiprows=9)
grade_2025 = pd.read_excel(file_2025_path, sheet_name="Grade", header=None, skiprows=6)

FileNotFoundError: [Errno 2] No such file or directory: 'PTSJ_data.xlsx'

## ⚙️ Define Constants & User Parameters

In [ ]:
palette = "tab10"
secondary_palette = "tab10"
colors = sns.color_palette(palette)
colors_sec = sns.color_palette(secondary_palette)
plot_theme = "whitegrid"

solid_density = 2.53            # Density of solids in slurry (g/cm³)
liquid_density = 1.15           # Density of liquid in slurry (g/cm³)
percent_solids_avg = 0.3391         # Solids in slurry (2024 avg. 33.91%)

# Switch for 2024 Throughput value
tp_2024 = True      # True assumes the unit is in wet m3 and False assumes it is in dry tonnes

# Leach circuit parameters
leach_tank_count = 2
cil_tank_count = 6
leach_tank_m3 = 1520
cil_tank_m3 = 568
circuit_volume_m3 = (leach_tank_count * leach_tank_m3) + (cil_tank_count * cil_tank_m3)

# Define "stable DO" threshold
stable_threshold = 7.5

figsize = (6.5, 4)

## 🧹 Data Cleansing & Preprocessing

In [ ]:
# --- Clean the 2024 dataframe ---
# Rename duplicate columns
def rename_duplicate_columns(cols):
	seen = {}
	new_cols = []
	for col in cols:
		if col not in seen:
			seen[col] = 1
			new_cols.append(col)
		else:
			seen[col] += 1
			new_cols.append(f"{col}_{seen[col]}")
	return new_cols

df_2024.columns = rename_duplicate_columns(df_2024.columns)

# Clean columns names
def clean_column_name(col):
	col = col.replace("%", "pct")
	col = col.replace("\n", "_")
	col = col.replace("(", "_").replace(")", "_")
	col = col.replace("_gram_", "_g_")
	col = col.replace("<", "").replace(">", "")
	col = col.replace("+", "plus").replace("-", "minus")
	col = re.sub(r"_+$", "", col)  # Remove trailing underscores
	col = col.replace(".", "_")
	col = col.replace("___", "_")
	col = col.replace("__", "_")  # Replace double underscores with single
	col = col.replace("Weight_Retained_Percentage_pct", "Weight_Retained_pct")
	return col.strip()

# Apply the cleaning function to all column names
df_2024.columns = [clean_column_name(col) for col in df_2024.columns]

# Calculate Slurry Volume in m³
if tp_2024 == True:
	df_2024["Slurry_Volume_m3"] = df_2024["Throughput"]
else:
	df_2024["Slurry_Volume_m3"] = df_2024["Throughput"] * (
		(df_2024["PercentSolids"] / 100 / (solid_density * 1000)) + 
		((1- df_2024["PercentSolids"] / 100) / (liquid_density * 1000))
	) * 1000

# Calculate residence time in hours
df_2024["Residence_Time_hr"] = circuit_volume_m3 / df_2024["Slurry_Volume_m3"] * 24

print(f"2024 DataFrame shape: {df_2024.shape}")

# --- Clean 2025 data ---
# Manual column mapping
col_names = {
	3: "Date", 4: "Time", 5: "Date Time",
	6: "Pre_Leach_Density_pct_w_w", 7: "Leach_Tank_01_Density_pct_w_w",
	8: "Leach_Tank_01_pH", 9: "Leach_Tank_01_CN_ppm", 10: "Leach_Tank_01_DO_ppm",
	11: "Leach_Tank_02_Density_pct_w_w", 12: "Leach_Tank_02_pH", 13: "Leach_Tank_02_CN_ppm", 14: "Leach_Tank_02_DO_ppm",
	15: "CIL_Tank_01_Density_pct_w_w", 16: "CIL_Tank_01_pH", 17: "CIL_Tank_01_CN_ppm", 18: "CIL_Tank_01_DO_ppm", 19: "CIL_Tank_01_Carbon_gpl",
	20: "CIL_Tank_02_Density_pct_w_w", 21: "CIL_Tank_02_pH", 22: "CIL_Tank_02_CN_ppm", 23: "CIL_Tank_02_DO_ppm", 24: "CIL_Tank_02_Carbon_gpl",
	25: "CIL_Tank_03_Density_pct_w_w", 26: "CIL_Tank_03_pH", 27: "CIL_Tank_03_CN_ppm", 28: "CIL_Tank_03_DO_ppm", 29: "CIL_Tank_03_Carbon_gpl",
	30: "CIL_Tank_04_Density_pct_w_w", 31: "CIL_Tank_04_pH", 32: "CIL_Tank_04_CN_ppm", 33: "CIL_Tank_04_DO_ppm", 34: "CIL_Tank_04_Carbon_gpl",
	35: "CIL_Tank_05_Density_pct_w_w", 36: "CIL_Tank_05_pH", 37: "CIL_Tank_05_CN_ppm", 38: "CIL_Tank_05_DO_ppm", 39: "CIL_Tank_05_Carbon_gpl",
	40: "CIL_Tank_06_Density_pct_w_w", 41: "CIL_Tank_06_pH", 42: "CIL_Tank_06_CN_ppm", 43: "CIL_Tank_06_DO_ppm", 44: "CIL_Tank_06_Carbon_gpl",
	45: "Tailing_Density_pct_w_w", 46: "Tailing_CN_ppm", 47: "DO_Purity_pct_w_w", 48: "DO_Flow",
}

# Clean each month's data and reassign cleaned versions
month_names = ["jan_2025", "feb_2025", "mar_2025", "apr_2025"]
df_2025_list = [jan_2025, feb_2025, mar_2025, apr_2025]
cleaned_list = []

for month_data in df_2025_list:
	# Rename columns
	month_data.rename(columns=col_names, inplace=True)

	# Select relevant columns (Date, Time, Date Time, and process data columns)
	month_data = month_data.iloc[:, 3:49]
	
	# Select relevant columns and reset index
	month_data = month_data.iloc[6:].reset_index(drop=True)
	
	# Ensure 'Date' column is in datetime format
	month_data['Date'] = pd.to_datetime(month_data['Date'], errors='coerce')

	cleaned_list.append(month_data)

# Reassign to original variable names
jan_2025, feb_2025, mar_2025, apr_2025 = cleaned_list

# Define month ranges and apply filtering
month_ranges = {
	"jan_2025": ("2025-01-01", "2025-01-31"),
	"feb_2025": ("2025-02-01", "2025-02-28"),
	"mar_2025": ("2025-03-01", "2025-03-31"),
	"apr_2025": ("2025-04-01", "2025-04-30"),
}

for name, (start, end) in month_ranges.items():
	df = locals()[name]
	df = df[(df["Date"] >= Timestamp(start)) & (df["Date"] <= Timestamp(end))]
	locals()[name] = df  # assign the filtered df back


# Display dataframe shapes for each month
print(f"January 2025 DataFrame shape: {jan_2025.shape}")
print(f"February 2025 DataFrame shape: {feb_2025.shape}")
print(f"March 2025 DataFrame shape: {mar_2025.shape}")
print(f"April 2025 DataFrame shape: {apr_2025.shape}")

# Combine the monthly data into a single DataFrame
df_2025 = pd.concat([jan_2025, feb_2025, mar_2025, apr_2025], ignore_index=True)

# Print the final shape of the 2025 DataFrame
print(f"Final 2025 DataFrame shape: {df_2025.shape}")

# --- Clean 2025 Grade data ---
# Assign manual column names to known first 14 columns based on inspection
manual_columns = [
	"Ignore",
	"Date",
	"Feed_Ton_Dry",
	"Feed_Grade_Au_ppm",
	"Feed_Grade_Ag_ppm",
	"Feed_Grade_Cu_ppm",
	"Tail_Ton_Dry",
	"Tail_Solution_Volume_m3",
	"Tail_Solid_Au_ppm",
	"Tail_Solid_Ag_ppm",
	"Tail_Solid_Cu_ppm",
	"Tail_Solution_Au_ppm",
	"Tail_Solution_Ag_ppm",
	"Tail_Solution_Cu_ppm"
]

# Apply to first 14 columns only
grade_2025 = grade_2025.iloc[:, :14]
grade_2025.columns = manual_columns

grade_2025["Slurry_Volume_m3"] = grade_2025["Feed_Ton_Dry"] * (
	(percent_solids_avg / (solid_density * 1000)) + 
	((1- percent_solids_avg) / (liquid_density * 1000))
	) * 1000

# Drop rows where 'Feed_Ton_Dry' is NaN or zero
grade_2025 = grade_2025.dropna(subset=["Feed_Ton_Dry"])
grade_2025 = grade_2025[grade_2025["Feed_Ton_Dry"] > 0]

grade_2025["Residence_Time_hr"] = circuit_volume_m3 / grade_2025["Slurry_Volume_m3"]

# Drop 'Ignore' column and rows without date
grade_2025 = grade_2025.drop(columns=["Ignore"]).dropna(subset=["Date"]).copy()
grade_2025["Date"] = pd.to_datetime(grade_2025["Date"], errors="coerce")

# Convert numeric columns
for col in grade_2025.columns:
	if col != "Date":
		grade_2025[col] = pd.to_numeric(grade_2025[col], errors="coerce")

# Preview final cleaned Grade sheet
print(f"Grade 2025 DataFrame shape: {grade_2025.shape}")

# Calculate gold input in grams
grade_2025["Au_Input_g"] = grade_2025["Feed_Ton_Dry"] * grade_2025["Feed_Grade_Au_ppm"]

# Calculate gold loss to tailings (solids only, in grams)
grade_2025["Au_Tail_g"] = grade_2025["Tail_Ton_Dry"] * grade_2025["Tail_Solid_Au_ppm"]

# Calculate recovery (%)
grade_2025["Recovery_pct"] = 100 * (1 - grade_2025["Au_Tail_g"] / grade_2025["Au_Input_g"])

# --- Prepare your merged DataFrame ---
# Group Final 2025 by date and take the mean for numeric fields and first for non-numeric fields
df_2025_daily = df_2025.groupby("Date", as_index=False).agg(
	lambda x: x.mean() if pd.api.types.is_numeric_dtype(x) else x.iloc[0])

# Merge on Date (inner to keep matching dates only)
merged_2025 = pd.merge(df_2025_daily, grade_2025, on="Date", how="inner")

# Convert key columns to numeric
merged_2025["Leach_Tank_01_CN_ppm"] = pd.to_numeric(merged_2025["Leach_Tank_01_CN_ppm"], errors="coerce")
merged_2025["Leach_Tank_01_DO_ppm"] = pd.to_numeric(merged_2025["Leach_Tank_01_DO_ppm"], errors="coerce")
merged_2025["Recovery_pct"] = pd.to_numeric(merged_2025["Recovery_pct"], errors="coerce")
merged_2025["Feed_Ton_Dry"] = pd.to_numeric(merged_2025["Feed_Ton_Dry"], errors="coerce")

# Derive CN added and CN in tailings (kg), and efficiency metrics
merged_2025["NaCN_Added_Est_kg"] = merged_2025["Leach_Tank_01_CN_ppm"] * merged_2025["Feed_Ton_Dry"] * 1e3 / 1e6

# Drop nan and - from Tailing_CN_ppm
merged_2025["Tailing_CN_ppm"] = merged_2025["Tailing_CN_ppm"].astype(str).replace("-", np.nan)
merged_2025["Tailing_CN_ppm"] = pd.to_numeric(merged_2025["Tailing_CN_ppm"], errors="coerce")
merged_2025["Tailing_CN_ppm"] = merged_2025["Tailing_CN_ppm"].replace(0, np.nan)
merged_2025 = merged_2025.dropna(subset=["Tailing_CN_ppm"])

# Display unique values in Tailing_CN_ppm after cleaning
df_2025.dropna(subset=["Tailing_CN_ppm"], inplace=True)

# Drop rows with zero or NaN Tailing_CN_ppm
merged_2025["NaCN_in_Tails_Est_kg"] = merged_2025["Tailing_CN_ppm"] * merged_2025["Tail_Solution_Volume_m3"] / 1000
merged_2025["NaCN_Loss_pct"] = 100 * merged_2025["NaCN_in_Tails_Est_kg"] / merged_2025["NaCN_Added_Est_kg"]
merged_2025["NaCN_Utilisation_pct"] = 100 - merged_2025["NaCN_Loss_pct"]

merged_2025["NaCN_Difference_ppm"] = merged_2025["Leach_Tank_01_CN_ppm"] - merged_2025["Tailing_CN_ppm"]
merged_2025["NaCN_Difference_kg"] = merged_2025["NaCN_Added_Est_kg"] - merged_2025["NaCN_in_Tails_Est_kg"]

print(f"Merged 2025 DataFrame shape: {merged_2025.shape}")


2024 DataFrame shape: (35, 49)
January 2025 DataFrame shape: (372, 46)
February 2025 DataFrame shape: (336, 46)
March 2025 DataFrame shape: (372, 46)
April 2025 DataFrame shape: (360, 46)
Final 2025 DataFrame shape: (1440, 46)
Grade 2025 DataFrame shape: (106, 15)
Merged 2025 DataFrame shape: (90, 69)


In [ ]:
df_2024.to_csv("df_2024.csv", index=False)
df_2025.to_csv("df_2025.csv", index=False)
grade_2025.to_csv("grade_2025.csv", index=False)
merged_2025.to_csv("merged_2025.csv", index=False)

## 📊 Define Plotting Helpers

In [ ]:
# --- Basic bar helper ---
def plot_bar(data, title, xlabel=None, ylabel=None, x_col=None, y_col=None, figsize=figsize,
			 color=colors[0], series=False, theme=plot_theme, palette=palette,
			 rotation=45, edge_color="black", horizontal=False, alpha=1.0):
	plt.figure(figsize=figsize)
	sns.set_theme(style=theme, palette=palette)
	if series:
		data.plot(kind='bar',color=color, edgecolor=edge_color,
			figsize=figsize, legend=False, alpha=alpha)
	else:
		sns.barplot(data=data, x=x_col, y=y_col, color=color, alpha=alpha,
			   orient='h' if horizontal else 'v')
	plt.title(title)
	plt.xlabel(xlabel, fontsize=10)
	plt.ylabel(ylabel, fontsize=10)
	if horizontal:
		plt.xticks(rotation=rotation)
	else:
		plt.xticks(rotation=rotation)
	plt.grid(False)
	plt.gca().spines['top'].set_visible(False)
	plt.gca().spines['right'].set_visible(False)
	plt.tight_layout()
	plt.show()


# --- Basic scatter helper ---
def plot_scatter(data, x_col, y_col, title, xlabel, ylabel, figsize=figsize, color=colors[0],
				 reg=False, theme=plot_theme, palette=palette, alpha=0.7, vertical_line=None):
	plt.figure(figsize=figsize)
	sns.set_theme(style=theme, palette=palette)
	sns.scatterplot(data=data, x=x_col, y=y_col, color=color)
	if reg:
		sns.regplot(data=data, x=x_col, y=y_col, scatter=False, color=color,
					line_kws={"linestyle": "--"})
	if vertical_line is not None:
		plt.axvline(x=vertical_line, color='red', linestyle='--', label='Vertical Line')
	plt.title(title)
	plt.xlabel(xlabel, fontsize=10)
	plt.ylabel(ylabel, fontsize=10)
	plt.grid(False)
	plt.tight_layout()
	plt.show()


# --- Scatter with regression line helper ---
def plot_scatter_with_regression(data, x_col, y_col, xlabel, ylabel, title=None, figsize=figsize,
								 color=colors[0], reg_color="red", order=1, ci=95, n_boot=1000,
								 ylim=None,
								 theme=plot_theme, palette=palette):
	plt.figure(figsize=figsize)
	sns.set_theme(style=theme, palette=palette)
	ax = plt.gca()

	sns.scatterplot(data=data, x=x_col, y=y_col, color=color)

	if pd.api.types.is_datetime64_any_dtype(data[x_col]):
		# --- Convert dates to numeric ---
		x_dates = mdates.date2num(data[x_col])
		y_vals = data[y_col].values
		mask = ~np.isnan(x_dates) & ~np.isnan(y_vals)
		x = x_dates[mask].reshape(-1, 1)
		y = y_vals[mask]

		# --- Polynomial Fit ---
		poly = PolynomialFeatures(order)
		x_poly = poly.fit_transform(x)
		model = LinearRegression().fit(x_poly, y)
		x_fit = np.linspace(x.min(), x.max(), 200).reshape(-1, 1)
		x_fit_poly = poly.transform(x_fit)
		y_fit = model.predict(x_fit_poly)

		# --- Confidence Interval via Bootstrap ---
		y_bootstrap = []
		for _ in range(n_boot):
			x_resample, y_resample = resample(x, y)
			x_poly_resample = poly.fit_transform(x_resample)
			model_boot = LinearRegression().fit(x_poly_resample, y_resample)
			y_bootstrap.append(model_boot.predict(x_fit_poly))

		y_bootstrap = np.array(y_bootstrap)
		lower = np.percentile(y_bootstrap, (100 - ci) / 2, axis=0)
		upper = np.percentile(y_bootstrap, 100 - (100 - ci) / 2, axis=0)

		# --- Plot regression line and confidence band ---
		ax.plot(mdates.num2date(x_fit.flatten()), y_fit, linestyle="--", color=reg_color)
		ax.fill_between(mdates.num2date(x_fit.flatten()), lower, upper, color=reg_color, alpha=0.2)
	else:
		# Use Seaborn for numeric x-axis
		sns.regplot(data=data, x=x_col, y=y_col, scatter=False, color=reg_color,
					line_kws={"linestyle": "--"}, order=order, ci=ci)

	if title:
		plt.title(title)
		
	plt.xlabel(xlabel, fontsize=10)
	plt.ylabel(ylabel, fontsize=10)
	
	if ylim:
		plt.ylim(ylim)

	if "Date" in x_col:
		ax.xaxis.set_major_formatter(mdates.DateFormatter("%d-%b"))
		plt.xticks(rotation=45)

	plt.grid(False)
	plt.tight_layout()
	plt.show()


# --- Line trend helper with cleaned numeric values ---
def plot_line_trend(df, y, yLabel, title="2025", figsize=figsize,
					theme=plot_theme, palette=palette, color=colors[0]):

	# Ensure col is in numeric format
	df[y] = pd.to_numeric(df[y], errors="coerce")

	if y not in df.columns:
		print(f"Column '{y}' not found.")
		return

	# Plot
	plt.figure(figsize=figsize)
	sns.set_theme(style=theme, palette=palette)
	ax = sns.lineplot(data=df.dropna(subset=[y]), x="Date", y=y, color=color)
	ax.spines["top"].set_visible(False)
	ax.spines["right"].set_visible(False)
	ax.set_title(title)
	ax.set_xlabel("Date")
	ax.set_ylabel(yLabel)
	
	# Better x-axis formatting
	ax.xaxis.set_major_locator(mdates.AutoDateLocator())
	ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
	plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
	plt.grid(False)
	plt.tight_layout()
	plt.show()


# --- Line plot with standard deviation band helper ---
def plot_line_with_std_band(df, x_col, y_col, title, xlabel, ylabel,
							color=colors[2], alpha=0.2,
							show_ref_line=True, ref_line_val=0, ref_line_color="red",
							ref_line_label="Reference", std_label="±1 Std Dev",
							figsize=figsize, theme=plot_theme, palette=palette):

	plt.figure(figsize=figsize)
	sns.set_theme(style=theme, palette=palette)

	ax = sns.lineplot(data=df, x=x_col, y=y_col, color=color, label=y_col)

	# ±1 standard deviation band
	std_dev = df[y_col].std()
	plt.fill_between(df[x_col],
					 df[y_col] - std_dev,
					 df[y_col] + std_dev,
					 color=color, alpha=alpha, label=std_label)

	# Optional reference line
	if show_ref_line:
		plt.axhline(ref_line_val, color=ref_line_color, linestyle='--', label=ref_line_label)

	plt.title(title)
	plt.xlabel(xlabel)
	plt.ylabel(ylabel)
	plt.xticks(rotation=45)
	plt.legend()
	plt.grid(False)
	plt.tight_layout()
	plt.show()



# --- Dual line plot helper ---
def plot_dual_lines(df1, df2, x_col, y_col1, y_col2, label1, label2,
					title="Dual Line Plot", xlabel="Date", ylabel="Value",
					color_1=colors[0], color_2=colors[1],
					figsize=figsize, theme=None, sec_axis=False):
	plt.figure(figsize=figsize)

	ax1 = plt.gca()
	line1 = sns.lineplot(data=df1, x=x_col, y=y_col1, ax=ax1, color=color_1)
	line1_line = line1.lines[0]

	if sec_axis:
		ax2 = ax1.twinx()
		line2 = sns.lineplot(data=df2, x=x_col, y=y_col2, ax=ax2, color=color_2)
		ax2.set_ylabel(f"{label2} ({ylabel})")
		line2_line = line2.lines[0]
	else:
		ax2 = ax1  # reuse ax1
		line2 = sns.lineplot(data=df2, x=x_col, y=y_col2, ax=ax1, color=color_2)
		line2_line = line2.lines[1]  # index 1 since it's the second line on the same axis

	ax1.set_title(title)
	ax1.set_xlabel(xlabel)
	ax1.set_ylabel(f"{label1} ({ylabel})")

	for ax in set([ax1, ax2]):  # avoid redundant styling if same axis
		ax.grid(False)
		ax.spines['top'].set_visible(False)
		ax.spines['right'].set_visible(False)

	ax1.tick_params(axis='x', rotation=45)

	ax1.legend(
		[line1_line, line2_line],
		[label1, label2],
		loc='upper left'
	)

	plt.tight_layout()
	plt.show()



# --- Time series panel plot helper ---
def plot_time_series_panels(
	panels,
	suptitle=None,
	sharex=True,
	figsize=(14, 10),
	date_format="%b %d",
	rotate_xticks=45,
	date_interval=1
):
	"""
	Plots multiple time series subplots stacked vertically.

	Parameters:
	- panels (list of dict): Each dict defines a subplot, with:
		{
			"title": str,
			"ylabel": str,
			"lines": [
				{
					"data": pd.DataFrame,
					"x": str,
					"y": str,
					"label": str,
					"color": str,
					"linestyle": str (optional),
					"linewidth": float (optional)
				},
				...
			]
		}
	- suptitle (str): Overall title for the figure.
	- sharex (bool): Whether subplots share the same x-axis.
	- figsize (tuple): Size of the figure.
	- date_format (str): Format for x-axis date labels.
	- rotate_xticks (int): Degrees to rotate x-tick labels.
	- date_interval (int): Interval in weeks for x-tick labels.
	"""
	n_panels = len(panels)
	fig, axs = plt.subplots(n_panels, 1, figsize=figsize, sharex=sharex)
	axs = axs if isinstance(axs, (list, np.ndarray)) else [axs]

	for ax, panel in zip(axs, panels):
		for line in panel["lines"]:
			sns.lineplot(
				data=line["data"],
				x=line["x"],
				y=line["y"],
				ax=ax,
				label=line["label"],
				color=line.get("color"),
				linestyle=line.get("linestyle", "-"),
				linewidth=line.get("linewidth", 2),
			)
		
		# Apply ylim if provided
		if "ylim" in panel:
			ax.set_ylim(*panel["ylim"])

		# Add vertical shaded spans
		for span in panel.get("vspans", []):
			# Convert start and end to datetime if not already
			start = pd.to_datetime(span["start"])
			end = pd.to_datetime(span["end"])
			ax.axvspan(start, end, color=span.get("color", "grey"), alpha=span.get("alpha", 0.1))

		# Add annotations
		for ann in panel.get("annotations", []):
			ax.annotate(
				ann["text"],
				xy=(ann["x"], ann["y"]),
				xytext=ann.get("xytext", (0, 10)),
				textcoords="offset points",
				arrowprops=dict(arrowstyle="->", lw=1),
				fontsize=ann.get("fontsize", 10),
				color=ann.get("color", "black")
			)

		ax.spines["top"].set_visible(False)
		ax.spines["right"].set_visible(False)
		ax.set_title(panel["title"])
		ax.set_ylabel(panel["ylabel"])
		ax.legend()
		ax.grid(False)

	# Format x-axis as dates
	for ax in axs:
		ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=date_interval))
		ax.xaxis.set_major_formatter(mdates.DateFormatter(date_format))
		ax.tick_params(axis="x", rotation=rotate_xticks)

	if suptitle:
		plt.suptitle(suptitle, fontsize=16, y=1.02)
		plt.subplots_adjust(top=0.93)
	plt.tight_layout()
	plt.show()


# --- Histogram helper ---
def plot_histogram(data, column, title, xlabel, ylabel="Frequency", figsize=figsize,
				   color=colors[0], bins=15, theme=plot_theme, palette=palette, kde=False):
	plt.figure(figsize=figsize)
	sns.set_theme(style=theme, palette=palette)
	sns.histplot(data[column], bins=bins, color=color, kde=kde)
	plt.title(title)
	plt.xlabel(xlabel)
	plt.ylabel(ylabel)
	plt.grid(False)
	plt.tight_layout()
	plt.show()


# --- Dual histogram helper ---
def plot_dual_histogram(data1, data2, column1, column2, title, xlabel, ylabel, label1, label2,
						figsize=figsize, color1=colors[0], color2=colors[4], theme=plot_theme,
						palette=palette):
	plt.figure(figsize=figsize)
	sns.set_theme(style=theme, palette=palette)
	sns.histplot(data1[column1], bins=15, color=color1, label=label1, kde=True, alpha=0.5)
	sns.histplot(data2[column2], bins=15, color=color2, label=label2, kde=True, alpha=0.5)
	plt.title(title)
	plt.xlabel(xlabel)
	plt.ylabel(ylabel)
	plt.legend()
	plt.grid(False)
	plt.tight_layout()
	plt.show()

# --- Histogram subplot helper ---
def plot_histogram_subplots(data, fields, ncols=2, bins=20, colors=None,
							kde=True, titles=None, xlabels=None, figsize=(14, 6)):
	"""
	Plot histograms with optional KDE for a list of numeric fields.

	Parameters:
	- data (pd.DataFrame): The DataFrame containing the data.
	- fields (list of str): List of column names to plot.
	- ncols (int): Number of subplots per row.
	- bins (int): Number of histogram bins.
	- colors (list of str): List of colours to use for each histogram.
	- kde (bool): Whether to overlay a KDE curve.
	- titles (list of str): Custom titles for each subplot (optional).
	- xlabel_suffix (str): Text to append to x-axis labels.
	- figsize (tuple): Size of the entire figure.
	"""
	n_fields = len(fields)
	nrows = math.ceil(n_fields / ncols)

	fig, axs = plt.subplots(nrows, ncols, figsize=figsize, squeeze=False)
	axs = axs.flatten()

	for i, field in enumerate(fields):
		color = colors[i] if colors and i < len(colors) else None
		title = titles[i] if titles and i < len(titles) else f"{field} Distribution"
		sns.histplot(data[field].dropna(), bins=bins, kde=kde, color=color, ax=axs[i])
		axs[i].set_title(title)
		if xlabels and i < len(xlabels):
			xlabel = xlabels[i]
		axs[i].set_xlabel(xlabel)
		axs[i].set_ylabel("Frequency")
		axs[i].grid(False)

	# Turn off unused subplots
	for j in range(n_fields, len(axs)):
		fig.delaxes(axs[j])

	plt.tight_layout()
	plt.show()

## Cyanide Loss Model
**Objective**
Predict cyanide consumption (kg/day or kg/tonne solids) as a function of process inputs and material characteristics.

Required Fields
Here’s a breakdown of essential fields typically needed for both empirical and mechanistic modelling:

**Target Variable**
* NaCN_Used_tonnes_per_day (or equivalent field)
* Or: Daily NaCN consumption (kg/t solids) (if pre-normalised)

**Material Feed Characteristics**
| Field                              | Description                               |
| ---------------------------------- | ----------------------------------------- |
| `Ore_Tonnes_Solids`                | Dry solids (t/day) entering leach circuit |
| `Au_Feed_Grade_gpt`                | Gold head grade (g/t)                     |
| `Ag_Feed_Grade_gpt`                | Silver head grade (if silver is relevant) |
| `Feed_Particle_Size`               | Average P80 or D50 (μm), if available     |
| `Solids_Percent` or `Pulp_Density` | % solids or slurry density                |

**Leach Tank Conditions**
| Field                     | Description                                                  |
| ------------------------- | ------------------------------------------------------------ |
| `CN_Concentration_Tank_X` | CN concentration (ppm) by tank or average across leach train |
| `Dissolved_Oxygen_Tank_X` | DO (ppm) by tank or average                                  |
| `pH_Tank_X`               | pH profile (relevant for CN stability)                       |
| `Temperature_Tank_X`      | Optional: temperature affects CN kinetics                    |

**Adsorption and Reagent Dynamics**
| Field                      | Description                      |
| -------------------------- | -------------------------------- |
| `Carbon_Concentration_gL`  | Activated carbon level (g/L)     |
| `Gold_Loading_on_Carbon`   | May correlate with CN efficiency |
| `Silver_Loading_on_Carbon` | Optional if silver is present    |
| `Carbon_Addition_Rate`     | May affect CN demand indirectly  |

**Lossess and Output Variables**
| Field                 | Description                                         |
| --------------------- | --------------------------------------------------- |
| `CN_Tailings_ppm`     | Free cyanide loss to tailings                       |
| `Gold_in_Tailings_gd` | Undissolved gold can correlate with CN inefficiency |
| `CN_Decomposition`    | Optional: inferred from pH + temperature trends     |

**Time-related Features**
| Field                 | Description                                           |
| --------------------- | ----------------------------------------------------- |
| `Date` or `Timestamp` | Useful for trend tracking, lag effects                |
| `Tank_Residence_Time` | Total or per tank if available                        |
| `Shift/Day Type`      | Operational factors (e.g. maintenance, rain, ramp-up) |



In [ ]:
df_2024_cols = df_2024.columns.tolist()
df_2025_cols = df_2025.columns.tolist()
merged_2025_cols = merged_2025.columns.tolist()

(df_2024.head, df_2024_cols), (df_2025.head, df_2025_cols), (merged_2025.head, merged_2025_cols)

## Assesement of coverage in the PTSJR Data
| Field Group                  | `df_2024`             | `df_2025`          | `merged_2025` (daily agg) |
| ---------------------------- | --------------------- | ------------------ | ------------------------- |
| **NaCN Added**               | ✅ `NaCN_Added`        | ⚠️ not explicit    | ✅ `NaCN_Added_Est_kg`     |
| **NaCN in Tailings**         | ✅ `NaCN_conc_tailing` | ✅ `Tailing_CN_ppm` | ✅ `NaCN_in_Tails_Est_kg`  |
| **Ore Throughput (t/d)**     | ✅ `Throughput`        | ❌                  | ✅ `Feed_Ton_Dry`          |
| **Gold Feed Grade**          | ✅ `Grade_Au`          | ❌                  | ✅ `Feed_Grade_Au_ppm`     |
| **Gold Tailings**            | ✅ `AU_tailing`        | ❌                  | ✅ `Tail_Solid_Au_ppm`     |
| **Particle Size / P80**      | ✅ `Au_P80_µm`         | ❌                  | ❌ `P80_um` constant only  |
| **% Solids / Pulp Density**  | ✅ `PercentSolids`     | ✅ (`_Density_pct`) | ❌ `Percent_Solids_pct` constant only |
| **Dissolved Oxygen (DO)**    | ❌                     | ✅ per tank         | ✅ tank-level + purity     |
| **CN Concentration (Tanks)** | ❌                     | ✅ per tank         | ✅ per tank                |
| **pH**                       | ❌                     | ✅ per tank         | ✅ per tank                |
| **Carbon Concentration**     | ❌                     | ✅ per tank         | ✅ per tank                |
| **Residence Time (hr)**      | ✅ `Residence_Time_hr` | ❌                  | ✅ `Residence_Time_hr`     |
